# 3. Линейные слои, функции активации и визуализация обучения

**Цель:** Реализовать полносвязные слои, изучить функции активации, построить классификатор и визуализировать обучение.

---

In [ ]:
import sys, os, logging, math
LOG_LEVEL = os.getenv("LOG_LEVEL", "DEBUG")
logging.basicConfig(level=getattr(logging, LOG_LEVEL), format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", stream=sys.stderr)
log = logging.getLogger("linear")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
log.info("Using device: %s", device)

## 3.1 Линейный слой (nn.Linear) изнутри

In [ ]:
log.debug("Inspecting nn.Linear internals")

linear = nn.Linear(in_features=4, out_features=3, bias=True)
print(f"Weight shape: {linear.weight.shape}")  # (3, 4)
print(f"Bias shape:   {linear.bias.shape}")    # (3,)
print(f"Weight:\n{linear.weight.data}")
print(f"Bias: {linear.bias.data}")

x = torch.randn(2, 4)
y = linear(x)
print(f"Input: {x.shape} -> Output: {y.shape}")
print(f"Output:\n{y}")

# Ручная реализация
y_manual = x @ linear.weight.T + linear.bias
print(f"Manual output matches: {torch.allclose(y, y_manual)}")

## 3.2 Функции активации

In [ ]:
log.debug("Visualizing activation functions")

x = torch.linspace(-5, 5, 200)

activations = {
    "ReLU": F.relu(x),
    "GELU": F.gelu(x),
    "Sigmoid": torch.sigmoid(x),
    "Tanh": torch.tanh(x),
    "LeakyReLU": F.leaky_relu(x, 0.1),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, y) in zip(axes.flat, activations.items()):
    ax.plot(x, y)
    ax.set_title(name)
    ax.grid(True)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
axes.flat[-1].axis('off')
plt.tight_layout()
plt.show()
log.info("Activation functions plotted")

In [ ]:
log.debug("Comparing GELU vs ReLU")

x = torch.linspace(-3, 3, 100)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(x, F.relu(x), label='ReLU')
plt.plot(x, F.gelu(x), label='GELU')
plt.legend()
plt.grid(True)
plt.title('ReLU vs GELU')

plt.subplot(1, 2, 2)
plt.plot(x, F.gelu(x) - F.relu(x))
plt.title('GELU - ReLU difference')
plt.grid(True)
plt.tight_layout()
plt.show()
log.info("GELU vs ReLU comparison shown")

## 3.3 Классификатор на синтетических данных

In [ ]:
log.debug("Generating synthetic dataset")

X_np, y_np = make_moons(n_samples=500, noise=0.15, random_state=42)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.long)

print(f"Dataset: X {X.shape}, y {y.shape}")
print(f"Classes: {y.unique().tolist()}")

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', alpha=0.7)
plt.title('Two Moons Dataset')
plt.grid(True)
plt.show()
log.info("Dataset generated: two moons, %d samples", len(X))

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=32, num_classes=2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = SimpleClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

X, y = X.to(device), y.to(device)
log.info("Model created: %s parameters", sum(p.numel() for p in model.parameters()))

In [ ]:
log.debug("Training classifier")

losses = []
accs = []
n_epochs = 200

for epoch in range(n_epochs):
    logits = model(X)
    loss = criterion(logits, y)
    
    optimizer.zero_grad()
    loss.backward()
    
    # Логируем градиенты
    if epoch % 50 == 0:
        grad_norm = sum(p.grad.norm().item() for p in model.parameters() if p.grad is not None)
        log.debug("Epoch %d: loss=%.4f, grad_norm=%.4f", epoch, loss.item(), grad_norm)
    
    optimizer.step()
    
    losses.append(loss.item())
    acc = (logits.argmax(1) == y).float().mean().item()
    accs.append(acc)

log.info("Training complete: final loss=%.4f, final acc=%.4f", losses[-1], accs[-1])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(losses)
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].grid(True)

# Accuracy curve
axes[1].plot(accs)
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].grid(True)

# Decision boundary
x_min, x_max = X[:, 0].min().item() - 0.5, X[:, 0].max().item() + 0.5
y_min, y_max = X[:, 1].min().item() - 0.5, X[:, 1].max().item() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32, device=device)
with torch.no_grad():
    Z = model(grid).argmax(1).cpu().numpy().reshape(xx.shape)
axes[2].contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
axes[2].scatter(X[:, 0].cpu(), X[:, 1].cpu(), c=y.cpu(), cmap='bwr', edgecolors='k', alpha=0.7)
axes[2].set_title('Decision Boundary')
axes[2].grid(True)

plt.tight_layout()
plt.show()
log.info("Training visualizations complete")

## 3.4 Сравнение активаций: ReLU vs GELU в классификаторе

In [ ]:
class ComparativeClassifier(nn.Module):
    def __init__(self, activation=F.relu, hidden_dim=64):
        super().__init__()
        self.activation = activation
        self.fc1 = nn.Linear(2, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 2)
        
    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        return self.fc3(x)

results = {}
for name, act in [("ReLU", F.relu), ("GELU", F.gelu)]:
    m = ComparativeClassifier(activation=act).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=0.01)
    losses = []
    for _ in range(200):
        l = nn.CrossEntropyLoss()(m(X), y)
        opt.zero_grad()
        l.backward()
        opt.step()
        losses.append(l.item())
    results[name] = losses
    log.info("%s classifier final loss: %.4f", name, losses[-1])

plt.figure(figsize=(8, 4))
for name, losses in results.items():
    plt.plot(losses, label=name)
plt.legend()
plt.title('ReLU vs GELU: Convergence')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

In [ ]:
print("=== Linear layers and activations complete ===")
print("Topics covered:")
print("  - nn.Linear internals (weight, bias, manual computation)")
print("  - Activation functions: ReLU, GELU, Sigmoid, Tanh, LeakyReLU")
print("  - GELU vs ReLU: why GELU is preferred in transformers")
print("  - Binary classifier on two moons dataset")
print("  - Training loop, loss curves, accuracy")
print("  - Decision boundary visualization")
print("  - Comparative convergence: ReLU vs GELU")
log.info("Linear layers notebook complete")